# Gdańsk Regional Championship: Day 2 team analysis

This notebook explores the 110 teams above the **Day 2 end** divider in the Gdańsk Regional Championship standings. It compares team compositions, groups teams connected by an exact similarity score, and demonstrates a damage matchup.

The Gdańsk dataset is stored at `m-c/gdansk_day_2.json`. Run this notebook with the `showcases` folder as the working directory. The `pyvgc` package must be installed in the active Python environment.

In [18]:
import json
from pyvgc import (
    group_identical_teamplayers,
    pairwise_score_matrix,
)

with open("m-c/regional-louisville-ky-day-2.json", encoding="utf-8") as data_file:
    standings_data = json.load(data_file)

teams = {
    entry["rank"]: {"player": entry["player"], "team": entry["team"]}
    for entry in standings_data[0:]
}
print(f"Loaded {len(teams)} teams ({sum(len(t['team']) for t in teams.values())} Pokémon sets).")
print("Standings source: https://standings.limitlessvgc.com/0044/standings")

Loaded 145 teams (870 Pokémon sets).
Standings source: https://standings.limitlessvgc.com/0044/standings


## Inspect the standings teams

Each record includes the player name and six sets containing Pokémon, item, ability, stat alignment (represented as nature by the package), and four moves.

In [19]:
for rank, entry in teams.items():
    roster = ", ".join(pokemon["name"] for pokemon in entry["team"])
    print(f"{rank:>3}. {entry['player']}: {roster}")

  1. Joseph Ugarte: Farigiraf, Incineroar, Charizard, Sylveon, Venusaur, Garchomp
  2. Andrew Figueras: Archaludon, Charizard, Pelipper, Venusaur, Grimmsnarl, Garchomp
  3. Dorian Kang: Venusaur, Politoed, Grimmsnarl, Raichu, Archaludon, Charizard
  4. JJ Miller: Gholdengo, Archaludon, Pelipper, Swampert, Staraptor, Sinistcha
  5. Ryan Loseto: Rillaboom, Kingambit, Sneasler, Eternal Flower Floette, Salamence, Basculegion
  6. Toler Webb: Golisopod, Garchomp, Pelipper, Archaludon, Farigiraf, Sneasler
  7. Keanu Inosanto: Basculegion, Rillaboom, Eternal Flower Floette, Sneasler, Salamence, Kingambit
  8. Brian Hough: Gholdengo, Dragonite, Incineroar, Sneasler, Rillaboom, Garchomp
  9. Paul Chua: Charizard, Sylveon, Incineroar, Venusaur, Toxapex, Garchomp
 10. Cary D'Ortona: Sneasler, Basculegion, Whimsicott, Garchomp, Charizard, Kingambit
 11. Justin Burns: Sneasler, Basculegion, Salamence, Kingambit, Whimsicott, Eternal Flower Floette
 12. Yichun Wu: Incineroar, Venusaur, Sylveon, Garch

## Compare team similarity

The score starts at 102 for two identical six-Pokémon teams.

More documentation about this function in the future.

A **higher score means more similar teams**. This implementation accepts the raw website team dictionaries, so it can also analyze sets that are outside the package's current M-C legality snapshot.

In [20]:
score_matrix = pairwise_score_matrix(teams)
column_width = max(4, len(str(max(teams))) + 1)

print("Rows and columns are standings ranks; identical teams score 102.")
print(" " * column_width + "".join(f"{rank:>{column_width}}" for rank in teams))
for rank, scores in score_matrix.items():
    print(f"{rank:>{column_width}}" + "".join(
        f"{scores[other_rank]:>{column_width}}" for other_rank in teams
    ))
print(f"Validated a symmetric {len(teams)} × {len(teams)} score matrix.")

Rows and columns are standings ranks; identical teams score 102.
       1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145
   1 102  49  32   0   0  26   0  32  77  30   0  77   0   0   0  42  15   0  30   0  43  17  26  59   0  14   0  15  40  94  57   0   0  77   0   0   0  61   0   0  26  14  11  15  29  77  14  31  15  15  15   0   0   0   0  26  13  14   0   0   0  77  28  26  16   0  78   0   0   0  49  16  43  25   0   0  32  34  14  26  15  43  26  38   0  26  

## Find score-linked groups

A pair is included only when its score equals the selected score. Groups are connected components: if A matches B and B matches C at that score.

## Exact-score pairs and player groups

Team comparison, pairwise matrix generation, exact-score pair lookup, and transitive player grouping are provided by the public `pyvgc` API. Grouping follows connected components: a chain of matching pairs joins its members even if not every pair within a group directly matches.

In [21]:
target_score_pairs = 102

score_groups = group_identical_teamplayers(score_matrix)
print("\nGroups of players with identical teams (score 102):")
for group in score_groups:
    members = ", ".join(
        f"{rank} ({teams[rank]['player']})" for rank in group
    )
    print(members)


Groups of players with identical teams (score 102):
7 (Keanu Inosanto), 18 (Cody Hruby), 27 (Nicholas Donnelly)
9 (Paul Chua), 12 (Yichun Wu), 62 (Junxi Zhu)
14 (Carson Confer), 35 (Zachary Weed), 93 (Marcus Dion)
15 (Wyatt Fout), 68 (Anthony Lugenbeal)
25 (Callie Murray), 92 (Christian Riewerts), 100 (Samad Ahmed)
33 (Kai Morton), 37 (Ryne Garrett)
125 (Jeffrey Lott), 132 (Huston Rhodes)


## Pokémon usage and set statistics

Pokémon usage is the percentage of these teams that brought each Pokémon. Nature, item, move, and exact-moveset percentages are calculated among that Pokémon's appearances. Teammate percentages are conditional: among teams that brought the selected Pokémon, the percentage that also brought each teammate. Forms are counted separately as named in the source data.

In [22]:
from collections import Counter, defaultdict

team_count = len(standings_data)
pokemon_team_counts = Counter()
sets_by_pokemon = defaultdict(list)
teammate_team_counts = defaultdict(Counter)

for entry in standings_data:
    members = entry["team"]
    pokemon_names = [member["name"] for member in members]
    if len(pokemon_names) != 6 or len(set(pokemon_names)) != 6:
        raise ValueError(
            f"Rank {entry['rank']} must have six distinct Pokémon"
        )
    pokemon_team_counts.update(pokemon_names)
    for member in members:
        sets_by_pokemon[member["name"]].append(member)
        teammate_team_counts[member["name"]].update(
            teammate for teammate in pokemon_names
            if teammate != member["name"]
        )

if sum(pokemon_team_counts.values()) != team_count * 6:
    raise ValueError("Expected exactly six Pokémon per team")


def print_table(title, headers, rows):
    formatted_rows = []
    for row in rows:
        formatted_rows.append(
            [
                f"{float(value):.1f}%" if header.endswith("%") else str(value)
                for header, value in zip(headers, row)
            ]
        )
    widths = [
        max(len(header), *(len(row[index]) for row in formatted_rows))
        for index, header in enumerate(headers)
    ]
    print(f"\n{title}")
    print(" | ".join(header.ljust(widths[index]) for index, header in enumerate(headers)))
    print("-+-".join("-" * width for width in widths))
    for row in formatted_rows:
        print(" | ".join(value.ljust(widths[index]) for index, value in enumerate(row)))

pokemon_order = sorted(
    pokemon_team_counts,
    key=lambda name: (-pokemon_team_counts[name], name.casefold()),
)


### Pokémon usage

In [23]:
usage_rows = [
    (
        pokemon,
        pokemon_team_counts[pokemon],
        pokemon_team_counts[pokemon] / team_count * 100,
    )
    for pokemon in pokemon_order
]
print_table(
    "Pokémon usage across Day 2 teams",
    ["Pokémon", "Teams", "Team usage %"],
    usage_rows,
)



Pokémon usage across Day 2 teams
Pokémon                | Teams | Team usage %
-----------------------+-------+-------------
Garchomp               | 64    | 44.1%       
Rillaboom              | 60    | 41.4%       
Incineroar             | 55    | 37.9%       
Sneasler               | 53    | 36.6%       
Kingambit              | 44    | 30.3%       
Raichu                 | 40    | 27.6%       
Charizard              | 36    | 24.8%       
Archaludon             | 35    | 24.1%       
Basculegion            | 35    | 24.1%       
Pelipper               | 28    | 19.3%       
Eternal Flower Floette | 23    | 15.9%       
Venusaur               | 23    | 15.9%       
Volcarona              | 23    | 15.9%       
Gholdengo              | 22    | 15.2%       
Farigiraf              | 20    | 13.8%       
Female Indeedee        | 18    | 12.4%       
Salamence              | 18    | 12.4%       
Sylveon                | 15    | 10.3%       
Delphox                | 14    | 9.7%        


### Nature, item, and movepool usage

Move usage counts each move once per set and reports the percentage of that Pokémon's sets containing it. Exact movesets are also listed as a four-move combination; their percentages sum to 100% for each Pokémon.

In [24]:
def distribution_rows(counter_by_pokemon):
    rows = []
    for pokemon in pokemon_order:
        denominator = pokemon_team_counts[pokemon]
        for option, count in sorted(
            counter_by_pokemon[pokemon].items(),
            key=lambda item: (-item[1], str(item[0]).casefold()),
        ):
            rows.append(
                (pokemon, option, count, count / denominator * 100)
            )
    return rows


nature_counts = defaultdict(Counter)
item_counts = defaultdict(Counter)
move_counts = defaultdict(Counter)
moveset_counts = defaultdict(Counter)
for pokemon, sets in sets_by_pokemon.items():
    for pokemon_set in sets:
        nature_counts[pokemon][pokemon_set.get("nature") or "Unknown"] += 1
        item_counts[pokemon][pokemon_set.get("item") or "Unknown"] += 1
        move_counts[pokemon].update(set(pokemon_set["moves"]))
        moveset = tuple(sorted(pokemon_set["moves"], key=str.casefold))
        moveset_counts[pokemon][moveset] += 1


def show_distribution(title, label, counter_by_pokemon, format_option=str):
    rows = [
        (pokemon, format_option(option), count, percent)
        for pokemon, option, count, percent in distribution_rows(
            counter_by_pokemon
        )
    ]
    print_table(
        title,
        ["Pokémon", label, "Sets", "Set usage %"],
        rows,
    )


show_distribution("Nature usage by Pokémon", "Nature", nature_counts)
show_distribution("Item usage by Pokémon", "Item", item_counts)
show_distribution("Individual move usage by Pokémon", "Move", move_counts)
show_distribution(
    "Exact four-move sets by Pokémon",
    "Moveset",
    moveset_counts,
    lambda moves: ", ".join(moves),
)



Nature usage by Pokémon
Pokémon                | Nature  | Sets | Set usage %
-----------------------+---------+------+------------
Garchomp               | Modest  | 23   | 35.9%      
Garchomp               | Adamant | 21   | 32.8%      
Garchomp               | Timid   | 11   | 17.2%      
Garchomp               | Jolly   | 8    | 12.5%      
Garchomp               | Careful | 1    | 1.6%       
Rillaboom              | Adamant | 52   | 86.7%      
Rillaboom              | Brave   | 3    | 5.0%       
Rillaboom              | Sassy   | 3    | 5.0%       
Rillaboom              | Careful | 2    | 3.3%       
Incineroar             | Careful | 23   | 41.8%      
Incineroar             | Impish  | 14   | 25.5%      
Incineroar             | Adamant | 12   | 21.8%      
Incineroar             | Sassy   | 3    | 5.5%       
Incineroar             | Relaxed | 2    | 3.6%       
Incineroar             | Calm    | 1    | 1.8%       
Sneasler               | Adamant | 39   | 73.6%      
Sne

### Teammate usage

In [25]:
teammate_rows = []
for pokemon in pokemon_order:
    denominator = pokemon_team_counts[pokemon]
    for teammate, count in sorted(
        teammate_team_counts[pokemon].items(),
        key=lambda item: (-item[1], item[0].casefold()),
    ):
        teammate_rows.append(
            (
                pokemon,
                teammate,
                count,
                count / denominator * 100,
            )
        )
print_table(
    "Teammates by Pokémon",
    ["Pokémon", "Teammate", "Teams together", "Teammate usage %"],
    teammate_rows,
)



Teammates by Pokémon
Pokémon                | Teammate               | Teams together | Teammate usage %
-----------------------+------------------------+----------------+-----------------
Garchomp               | Incineroar             | 33             | 51.6%           
Garchomp               | Charizard              | 26             | 40.6%           
Garchomp               | Rillaboom              | 25             | 39.1%           
Garchomp               | Sneasler               | 19             | 29.7%           
Garchomp               | Venusaur               | 19             | 29.7%           
Garchomp               | Archaludon             | 16             | 25.0%           
Garchomp               | Kingambit              | 16             | 25.0%           
Garchomp               | Raichu                 | 14             | 21.9%           
Garchomp               | Sylveon                | 14             | 21.9%           
Garchomp               | Farigiraf              | 13  